# Méthodes d'apprentissage automatique classiques

Notebook d'appui à la rédaction de la section « Méthodes » : principes, avantages et limites
des méthodes d'apprentissage automatique « classiques » (non neuronales) :

1. **Arbres de décision** (brique de base des méthodes d'ensemble) ;
2. **Forêts aléatoires** (*Random Forest*) ;
3. **Gradient boosting** : XGBoost et LightGBM ;
4. **Machines à vecteurs de support** (SVM) ;
5. **Comparaison** des surfaces de probabilité et de la **calibration**.

L'objectif est de **décrire** les méthodes. Les illustrations utilisent les vraies données de la
LNH avec quelques variables simples connues avant le match (`nhl_forecast.features.jeu_base`),
des hyperparamètres raisonnables et **aucune optimisation** : la comparaison rigoureuse des
modèles (variables, réglage, validation temporelle) est faite dans `03_experimentation/`.

Avec `SAUVER = True`, les figures sont enregistrées en PNG (300 dpi) dans
`reports/map6009/figures/` et les tableaux en CSV dans `reports/map6009/tables/`
(préfixe `methodes_`).

In [ ]:
import lightgbm as lgb
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from matplotlib.patches import FancyBboxPatch, Patch
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

from nhl_forecast.config import PROJECT_ROOT
from nhl_forecast.features.jeu_base import DESCRIPTIONS, charger_jeu_base
from nhl_forecast.visualization.style import COULEURS, ENCRE, ENCRE_2, appliquer_style

In [ ]:
SAUVER = True  # True : enregistre aussi les figures (PNG) et tableaux (CSV) pour le rapport
FIG_DIR = PROJECT_ROOT / "reports" / "map6009" / "figures"
TAB_DIR = PROJECT_ROOT / "reports" / "map6009" / "tables"

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)
appliquer_style()  # couleurs + traits/marqueurs/hachures : lisible en noir et blanc
rng = np.random.default_rng(6009)


def sauver(fig, nom):
    if SAUVER:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{nom}.png", dpi=300, bbox_inches="tight", facecolor="white")


def sauver_tableau(df, nom):
    if SAUVER:
        TAB_DIR.mkdir(parents=True, exist_ok=True)
        df.to_csv(TAB_DIR / f"{nom}.csv", encoding="utf-8")
    return df

## Données utilisées

Un match par ligne ; cible : victoire de l'équipe locale. Treize variables, toutes des écarts
local − visiteur calculés sur les matchs précédents (saison à date, 10 derniers matchs, repos,
gardien partant). Seuls les matchs où toutes les variables sont disponibles sont conservés.
Séparation **temporelle** : entraînement 2008-2009 à 2020-2021, validation 2021-2022 (arrêt
précoce, calibration de Platt), test 2022-2023 à 2025-2026.

In [ ]:
VARS = [
    "d_pts_pct_saison", "d_diff_buts_saison", "d_tirs_pct_saison", "d_pp_pct_saison",
    "d_pk_pct_saison", "d_xg_pct_saison", "d_corsi_pct_saison", "d_xg_pour_saison",
    "d_xg_contre_saison", "d_pts_pct_10", "d_xg_pct_10", "d_repos", "d_partant_sv_saison",
]  # fmt: skip
COURTS = {v: v.removeprefix("d_") for v in VARS}  # noms courts pour les schémas d'arbre

matchs, _ = charger_jeu_base()
jeu = matchs[matchs.saison >= 2008].dropna(subset=VARS).reset_index(drop=True)
parts = {
    "entraînement": jeu.saison <= 2020,
    "validation": jeu.saison == 2021,
    "test": jeu.saison >= 2022,
}
X_tr, y_tr = jeu.loc[parts["entraînement"], VARS], jeu.loc[parts["entraînement"], "victoire_locale"]
X_va, y_va = jeu.loc[parts["validation"], VARS], jeu.loc[parts["validation"], "victoire_locale"]
X_te, y_te = jeu.loc[parts["test"], VARS], jeu.loc[parts["test"], "victoire_locale"]
P_CONST = y_tr.mean()
LL_CONST = log_loss(y_te, np.full(len(y_te), P_CONST))
print(pd.Series({k: int(m.sum()) for k, m in parts.items()}, name="matchs").to_string())
print(f"Log loss d'une probabilité constante ({P_CONST:.3f}) sur le test : {LL_CONST:.4f}")

---
## 1. Arbres de décision

**Principe.** Un arbre partitionne récursivement l'espace des variables par des coupures
binaires $x_j \le s$ choisies pour maximiser la pureté des nœuds enfants (indice de Gini ou
entropie). La probabilité prédite dans une feuille est la proportion de victoires parmi les
matchs d'entraînement qui y tombent. Un arbre seul est **instable** (forte variance) et produit
des probabilités en **escalier** ; il sert surtout de brique de base aux méthodes d'ensemble.

Arbre de profondeur 3 appris sur les 13 variables (au moins 200 matchs par feuille) :

In [ ]:
arbre = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=0).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(11, 4.8))
plot_tree(
    arbre,
    feature_names=[COURTS[v] for v in VARS],
    class_names=["Défaite", "Victoire"],
    impurity=False,
    proportion=True,
    filled=False,
    rounded=True,
    precision=3,
    fontsize=7,
    ax=ax,
)
for t in ax.texts:  # libellés en français
    t.set_text(
        t.get_text()
        .replace("samples", "obs.")
        .replace("value", "prop.")
        .replace("class", "classe")
        .replace("True", "Vrai")
        .replace("False", "Faux")
    )
sauver(fig, "methodes_arbre_structure")
plt.show()
print("Variables (noms courts = écart local − visiteur) :")
for v in VARS:
    print(f"  {COURTS[v]:<18} {DESCRIPTIONS[v]}")

Avec deux variables (écart de xG% de la saison et écart de % des points sur 10 matchs), la
partition se dessine dans le plan : chaque rectangle est une feuille, la probabilité y est
**constante** et les frontières sont toujours parallèles aux axes.

In [ ]:
V2 = ["d_xg_pct_saison", "d_pts_pct_10"]
ETIQ2 = [
    "Écart de xG%, saison à date (points de %)",
    "Écart de % des points, 10 derniers matchs (points de %)",
]
arbre2 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=0).fit(
    100 * X_tr[V2], y_tr
)
lim = np.quantile(100 * X_tr[V2], [0.01, 0.99], axis=0)
XX, YY = np.meshgrid(np.linspace(lim[0, 0], lim[1, 0], 241), np.linspace(lim[0, 1], lim[1, 1], 241))
GRILLE = pd.DataFrame(np.c_[XX.ravel(), YY.ravel()], columns=V2)
P_arbre = arbre2.predict_proba(GRILLE)[:, 1].reshape(XX.shape)
feuilles = arbre2.apply(GRILLE).reshape(XX.shape)

fig, ax = plt.subplots(figsize=(5.8, 4.6))
im = ax.pcolormesh(XX, YY, P_arbre, cmap="Greys", vmin=0.25, vmax=0.8, shading="auto")
ax.contour(XX, YY, feuilles, levels=np.unique(feuilles) + 0.5, colors=ENCRE, linewidths=0.8)
for f in np.unique(feuilles):
    m = feuilles == f
    pf = P_arbre[m][0]
    ax.text(
        XX[m].mean(),
        YY[m].mean(),
        f"{pf:.2f}".replace(".", ","),
        ha="center",
        va="center",
        fontsize=8,
        color="white" if pf > 0.6 else ENCRE,
    )
ax.set(xlabel=ETIQ2[0], ylabel=ETIQ2[1])
fig.colorbar(im, ax=ax, label="Probabilité prédite")
sauver(fig, "methodes_arbre_partition")
plt.show()

---
## 2. Forêts aléatoires

**Principe** (Breiman, 2001). On entraîne $B$ arbres profonds, chacun sur un **échantillon
bootstrap** des données et en ne considérant, à chaque coupure, qu'un **sous-ensemble aléatoire
de variables** (`max_features`). La probabilité prédite est la moyenne des arbres :

$$\hat p(\mathbf x) = \frac{1}{B} \sum_{b=1}^{B} \hat p_b(\mathbf x).$$

La moyenne d'arbres peu corrélés réduit fortement la **variance** sans augmenter le biais.
L'erreur hors sac (*out-of-bag*) donne une estimation de validation gratuite.

Ajouter des arbres n'entraîne pas de surapprentissage : la log loss diminue puis se stabilise.
La complexité est plutôt contrôlée par la taille minimale des feuilles et `max_features`. Sur des
données aussi bruitées que des résultats de hockey, des feuilles **grandes** sont nécessaires
pour obtenir des probabilités raisonnables.

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
ll_lr = log_loss(y_te, lr.predict_proba(X_te)[:, 1])

n_arbres = [1, 2, 5, 10, 20, 50, 100, 200, 500]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for feuille, c, ls, mk in [(5, COULEURS[1], "--", "s"), (100, COULEURS[0], "-", "o")]:
    rf_c = RandomForestClassifier(
        min_samples_leaf=feuille, max_features="sqrt", warm_start=True, random_state=0, n_jobs=-1
    )
    ll = []
    for b in n_arbres:
        rf_c.set_params(n_estimators=b).fit(X_tr, y_tr)
        ll.append(log_loss(y_te, np.clip(rf_c.predict_proba(X_te)[:, 1], 0.01, 0.99)))
    ax.plot(
        n_arbres,
        ll,
        color=c,
        ls=ls,
        marker=mk,
        label=f"Forêt aléatoire, feuilles ≥ {feuille} matchs",
    )
ax.axhline(ll_lr, color=ENCRE, ls=":", marker="", lw=1.2, label="Régression logistique")
ax.axhline(LL_CONST, color=ENCRE_2, ls="-.", marker="", lw=1, label="Probabilité constante")
ax.set_xscale("log")
ax.set_ylim(top=0.75)
ax.set_xlabel("Nombre d'arbres $B$")
ax.set_ylabel("Log loss (test)")
ax.legend(fontsize=8)
sauver(fig, "methodes_rf_nombre_arbres")
plt.show()

**Avantages.** Capte non-linéarités et interactions sans les spécifier ; robuste au bruit, aux
variables non pertinentes et aux échelles différentes ; peu d'hyperparamètres sensibles ;
parallélisable ; importance des variables disponible.

**Limites.** Probabilités souvent **mal calibrées** : la moyenne d'arbres ramène les
probabilités vers le centre et les extrêmes sont rarement atteints (Niculescu-Mizil et Caruana,
2005) ; avec de petites feuilles sur des données bruitées, au contraire, les arbres mémorisent
le bruit ; d'où le recours à la recalibration (Platt, isotonique). Modèle volumineux ; faible
extrapolation hors du domaine observé ; aucune notion de temps (variables temporelles à
construire).

---
## 3. Gradient boosting : XGBoost et LightGBM

**Principe** (Friedman, 2001). Le modèle est une somme d'arbres peu profonds construits
**séquentiellement**, chacun corrigeant les erreurs des précédents. À l'étape $m$, l'arbre
$h_m$ est ajusté au gradient négatif de la perte (pour la log loss : $y - \hat p$) et ajouté avec
un **taux d'apprentissage** $\eta$ :

$$F_m(\mathbf x) = F_{m-1}(\mathbf x) + \eta\, h_m(\mathbf x), \qquad
\hat p(\mathbf x) = \sigma\big(F_M(\mathbf x)\big).$$

**XGBoost** (Chen et Guestrin, 2016) utilise une approximation de second ordre de la perte
(gradient et hessien) et une régularisation explicite de la complexité des arbres,
$\Omega(h) = \gamma T + \tfrac12 \lambda \lVert \mathbf w \rVert^2$ ($T$ feuilles, poids
$\mathbf w$). **LightGBM** (Ke et al., 2017) vise la rapidité sur de grands jeux de données :
histogrammes de variables, croissance des arbres **par feuille** (*leaf-wise*),
échantillonnage GOSS et regroupement de variables exclusives (EFB).

Contrairement à la forêt aléatoire, le boosting **surapprend** si on ajoute trop d'arbres : le
nombre d'itérations se choisit par **arrêt précoce** sur un ensemble de validation. Un $\eta$
plus petit demande plus d'arbres, mais atteint souvent une meilleure perte. Sur des données à
signal faible, le surapprentissage arrive très vite.

In [ ]:
N_IT = 800
fig, ax = plt.subplots(figsize=(7, 3.4))
for eta, c, ls, mk in [
    (0.3, COULEURS[1], "--", "s"),
    (0.1, COULEURS[0], "-", "o"),
    (0.02, COULEURS[2], ":", "^"),
]:
    m = xgb.XGBClassifier(
        n_estimators=N_IT,
        learning_rate=eta,
        max_depth=3,
        eval_metric="logloss",
        random_state=0,
        n_jobs=-1,
    )
    m.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)
    res = m.evals_result()
    it = np.arange(1, N_IT + 1)
    va = np.array(res["validation_1"]["logloss"])
    ax.plot(it, va, color=c, ls=ls, marker=mk, markevery=80, label=f"Validation, $\\eta = {eta}$")
    if eta == 0.1:
        ax.plot(
            it,
            res["validation_0"]["logloss"],
            color=c,
            ls="-",
            lw=0.8,
            marker="",
            alpha=0.7,
            label=f"Entraînement, $\\eta = {eta}$",
        )
    i_min = int(np.argmin(va))
    ax.plot(it[i_min], va[i_min], marker="*", ms=11, color=ENCRE, ls="")
ax.plot([], [], marker="*", ms=11, color=ENCRE, ls="", label="Arrêt précoce (minimum)")
ax.set_ylim(0.63, 0.70)
ax.set_xlabel("Nombre d'arbres (itérations de boosting)")
ax.set_ylabel("Log loss")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
sauver(fig, "methodes_gb_taux_apprentissage")
plt.show()

**Croissance des arbres.** XGBoost développe par défaut les arbres **niveau par niveau**
(*level-wise*, profondeur maximale), alors que LightGBM développe à chaque étape la **feuille**
qui réduit le plus la perte (*leaf-wise*, nombre maximal de feuilles). La croissance par feuille
converge plus vite mais produit des arbres asymétriques, plus sujets au surapprentissage sur de
petits jeux de données (d'où `num_leaves` et `min_child_samples`). Les deux panneaux se lisent
en parallèle (les numéros indiquent l'ordre des coupures). Schéma conceptuel.

In [ ]:
def dessiner_arbre(ax, noeuds, aretes, titre):
    """noeuds : {id: (x, y, ordre ou None)} ; ordre None = feuille."""
    for a, b in aretes:
        (xa, ya, _), (xb, yb, _) = noeuds[a], noeuds[b]
        ax.plot([xa, xb], [ya, yb], color=ENCRE_2, lw=1, marker="", ls="-", zorder=1)
    for x, y, ordre in noeuds.values():
        if ordre is None:
            ax.add_patch(
                FancyBboxPatch(
                    (x - 0.22, y - 0.16),
                    0.44,
                    0.32,
                    boxstyle="round,pad=0.02",
                    fc="white",
                    ec=ENCRE,
                    hatch="///",
                    lw=0.8,
                    zorder=2,
                )
            )
        else:
            ax.add_patch(
                plt.Circle((x, y), 0.24, fc=COULEURS[0], ec=ENCRE, lw=0.8, zorder=2, alpha=0.9)
            )
            ax.text(
                x,
                y,
                str(ordre),
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                fontweight="bold",
                zorder=3,
            )
    ax.set_title(titre, fontsize=10)
    ax.set_xlim(-0.2, 8.2)
    ax.set_ylim(-2.6, 3.6)
    ax.set_aspect("equal")
    ax.axis("off")


# Niveau par niveau : arbre complet de profondeur 3 (7 coupures, 8 feuilles)
niv = {
    "r": (4, 3, 1),
    "a": (2, 2, 2),
    "b": (6, 2, 3),
    "aa": (1, 1, 4),
    "ab": (3, 1, 5),
    "ba": (5, 1, 6),
    "bb": (7, 1, 7),
}
niv |= {
    f"{k}{s}": (niv[k][0] + d, 0, None)
    for k in ["aa", "ab", "ba", "bb"]
    for s, d in [("g", -0.5), ("d", 0.5)]
}
aretes_niv = [("r", "a"), ("r", "b"), ("a", "aa"), ("a", "ab"), ("b", "ba"), ("b", "bb")]
aretes_niv += [(k, f"{k}{s}") for k in ["aa", "ab", "ba", "bb"] for s in "gd"]

# Par feuille : la feuille la plus utile est développée à chaque étape (7 coupures, 8 feuilles)
feu = {
    "r": (2.5, 3, 1),
    "a": (1, 2, None),
    "b": (4, 2, 2),
    "ba": (3, 1, 4),
    "bb": (6, 1, 3),
    "baa": (2.4, 0, None),
    "bab": (3.6, 0, None),
    "bba": (5, 0, 5),
    "bbb": (7.2, 0, 6),
    "bbaa": (4.5, -1, None),
    "bbab": (5.5, -1, None),
    "bbba": (6.6, -1, 7),
    "bbbb": (7.8, -1, None),
    "bbbaa": (6.1, -2, None),
    "bbbab": (7.1, -2, None),
}
aretes_feu = [
    ("r", "a"),
    ("r", "b"),
    ("b", "ba"),
    ("b", "bb"),
    ("ba", "baa"),
    ("ba", "bab"),
    ("bb", "bba"),
    ("bb", "bbb"),
    ("bba", "bbaa"),
    ("bba", "bbab"),
    ("bbb", "bbba"),
    ("bbb", "bbbb"),
    ("bbba", "bbbaa"),
    ("bbba", "bbbab"),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
dessiner_arbre(axes[0], niv, aretes_niv, "Croissance par niveau (XGBoost)")
dessiner_arbre(axes[1], feu, aretes_feu, "Croissance par feuille (LightGBM)")
axes[0].legend(
    handles=[
        plt.Line2D(
            [],
            [],
            marker="o",
            ms=10,
            ls="",
            color=COULEURS[0],
            mec=ENCRE,
            label="Nœud de coupure (ordre)",
        ),
        Patch(fc="white", ec=ENCRE, hatch="///", label="Feuille"),
    ],
    loc="lower center",
    ncols=2,
    fontsize=8,
    frameon=False,
)
fig.tight_layout()
sauver(fig, "methodes_gb_croissance_arbres")
plt.show()

**Avantages.** Souvent la meilleure performance sur données tabulaires ; capte non-linéarités
et interactions ; gère les valeurs manquantes nativement ; régularisation riche (profondeur,
$\eta$, sous-échantillonnage, $\lambda$, $\gamma$) ; rapide (LightGBM en particulier) ;
optimise directement la log loss, donc probabilités en général raisonnablement calibrées
après arrêt précoce.

**Limites.** Nombreux hyperparamètres interdépendants ; risque de surapprentissage sur petits
jeux de données (une saison LNH ≈ 1 300 matchs) et sur signal faible ; calibration à vérifier
(et à corriger au besoin) ; interprétation indirecte (importance, SHAP) ; pas de dynamique
temporelle native.

---
## 4. Machines à vecteurs de support (SVM)

**Principe** (Cortes et Vapnik, 1995). Le SVM cherche l'hyperplan
$f(\mathbf x) = \mathbf w^\top \mathbf x + b = 0$ qui **maximise la marge** entre les deux
classes. Avec marge souple, il minimise

$$\tfrac12 \lVert \mathbf w \rVert^2 + C \sum_i \max\big(0,\ 1 - y_i f(\mathbf x_i)\big),
\qquad y_i \in \{-1, 1\}.$$

Seuls les points sur la marge ou du mauvais côté (**vecteurs de support**) déterminent la
solution. L'**astuce du noyau** (ex. RBF, $k(\mathbf x, \mathbf x') = e^{-\gamma\lVert\mathbf x
- \mathbf x'\rVert^2}$) permet des frontières non linéaires.

SVM linéaire sur deux variables, 400 matchs d'entraînement tirés au hasard. Les classes se
chevauchent tellement que **presque tous les matchs** sont dans la marge ou du mauvais côté : la
notion de marge, pensée pour des classes séparables, est peu informative pour des résultats
sportifs.

In [ ]:
ech = jeu[parts["entraînement"]].sample(400, random_state=3)
Xs, ys = 100 * ech[V2].to_numpy(), ech["victoire_locale"].to_numpy()
svm_lin = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)).fit(Xs, ys)
xx, yy = np.meshgrid(np.linspace(*lim[:, 0], 300), np.linspace(*lim[:, 1], 300))
f = svm_lin.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
sv = Xs[svm_lin[-1].support_]

fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.contour(
    xx, yy, f, levels=[-1, 0, 1], colors=ENCRE, linestyles=["--", "-", "--"], linewidths=[1, 1.6, 1]
)
ax.contourf(xx, yy, f, levels=[-1, 1], colors="none", hatches=["...."])
ax.scatter(*sv.T, s=60, facecolor="none", edgecolor=ENCRE_2, lw=0.6, label="Vecteurs de support")
ax.scatter(
    *Xs[ys == 1].T,
    s=12,
    marker="o",
    color=COULEURS[0],
    edgecolor="white",
    lw=0.3,
    label="Victoire locale",
)
ax.scatter(*Xs[ys == 0].T, s=14, marker="x", color=COULEURS[1], lw=0.9, label="Défaite locale")
ax.set(xlabel=ETIQ2[0], ylabel=ETIQ2[1], xlim=lim[:, 0], ylim=lim[:, 1])
ax.legend(fontsize=8, loc="lower right", framealpha=0.95)
sauver(fig, "methodes_svm_marge")
plt.show()
print(f"{len(sv)} vecteurs de support sur {len(Xs)} matchs ({len(sv) / len(Xs):.0%})")

Le SVM produit un **score** $f(\mathbf x)$, pas une probabilité. Il faut une étape de
calibration, typiquement la **mise à l'échelle de Platt** : $\hat p = \sigma(a f(\mathbf x) + b)$,
avec $a, b$ estimés sur des données de validation. Exemple : SVM linéaire sur les 13 variables ;
les points sont les proportions observées de victoires locales (validation) par vingtile du
score.

In [ ]:
svm13 = make_pipeline(StandardScaler(), LinearSVC(C=0.01)).fit(X_tr, y_tr)
score_va = svm13.decision_function(X_va)
platt = LogisticRegression(C=1e6).fit(score_va.reshape(-1, 1), y_va)
q = pd.qcut(score_va, 20, labels=False)
emp = pd.DataFrame({"s": score_va, "y": y_va.to_numpy(), "q": q}).groupby("q").mean()
sg = np.linspace(score_va.min(), score_va.max(), 300)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(
    emp.s,
    emp.y,
    ls="",
    marker="s",
    ms=5,
    color=COULEURS[1],
    label="Proportion observée (vingtiles du score)",
)
ax.plot(
    sg,
    platt.predict_proba(sg.reshape(-1, 1))[:, 1],
    color=COULEURS[0],
    ls="-",
    marker="",
    label=f"Platt : $\\sigma({platt.coef_[0, 0]:.2f}\\,f {platt.intercept_[0]:+.2f})$",
)
ax.axvline(0, color=ENCRE_2, lw=0.8, ls=":")
ax.text(
    0.03, 0.05, "Frontière $f = 0$", transform=ax.get_xaxis_transform(), fontsize=8, color=ENCRE_2
)
ax.set_xlabel("Score du SVM $f(\\mathbf{x})$ (validation)")
ax.set_ylabel("$P(\\text{victoire locale})$")
ax.legend(fontsize=8, loc="upper left")
sauver(fig, "methodes_svm_platt")
plt.show()

**Avantages.** Fondement théorique solide (maximisation de la marge) ; efficace en grande
dimension ; noyaux flexibles ; solution unique (problème convexe).

**Limites.** Aucune probabilité native (calibration de Platt obligatoire, qui demande des
données supplémentaires) ; critère de marge peu adapté à des classes très chevauchantes ; coût
en $O(n^2)$ à $O(n^3)$ avec noyau ; choix du noyau et de $(C, \gamma)$ sensible ;
standardisation des variables nécessaire ; peu interprétable avec noyau non linéaire.

---
## 5. Comparaison des surfaces de probabilité

Les cinq modèles sont entraînés sur les **deux mêmes variables** (matchs d'entraînement). Le
premier panneau montre la proportion observée de victoires locales sur les matchs de test, par
case (cases d'au moins 30 matchs). Les six panneaux se lisent en parallèle : niveaux de gris =
probabilité, trait plein = isoprobabilité 0,5, tirets = 0,4 et 0,6. La log loss de test (deux
variables) est indiquée.

In [ ]:
X2_tr, X2_va, X2_te = 100 * X_tr[V2], 100 * X_va[V2], 100 * X_te[V2]


def ajuster(X_a, X_v):
    """Les cinq modèles, entraînés sur X_a (et validés sur X_v pour l'arrêt précoce et Platt)."""
    m = {
        "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression()).fit(
            X_a, y_tr
        ),
        "Forêt aléatoire": RandomForestClassifier(
            n_estimators=500, min_samples_leaf=100, max_features="sqrt", random_state=0, n_jobs=-1
        ).fit(X_a, y_tr),
    }
    m["XGBoost"] = xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=3,
        eval_metric="logloss",
        early_stopping_rounds=100,
        random_state=0,
        n_jobs=-1,
    )
    m["XGBoost"].fit(X_a, y_tr, eval_set=[(X_v, y_va)], verbose=False)
    m["LightGBM"] = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=8,
        min_child_samples=100,
        random_state=0,
        verbose=-1,
    )
    m["LightGBM"].fit(
        X_a, y_tr, eval_X=X_v, eval_y=y_va, callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    # SVM à noyau RBF (coût quadratique : 4 000 matchs d'entraînement) + Platt sur la validation
    idx = rng.choice(len(X_a), 4000, replace=False)
    svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0)).fit(
        X_a.iloc[idx], y_tr.iloc[idx]
    )
    m["SVM (RBF + Platt)"] = CalibratedClassifierCV(FrozenEstimator(svm), method="sigmoid").fit(
        X_v, y_va
    )
    return m


def proba(m, X):
    return m.predict_proba(X)[:, 1]


modeles2 = ajuster(X2_tr, X2_va)

# Proportion observée par case (test)
bx = np.linspace(*lim[:, 0], 13)
by = np.linspace(*lim[:, 1], 13)
cnt, _, _ = np.histogram2d(X2_te.iloc[:, 0], X2_te.iloc[:, 1], bins=[bx, by])
som, _, _ = np.histogram2d(X2_te.iloc[:, 0], X2_te.iloc[:, 1], bins=[bx, by], weights=y_te)
obs = np.where(cnt >= 30, som / np.maximum(cnt, 1), np.nan).T

fig, axes = plt.subplots(2, 3, figsize=(10.5, 7), sharex=True, sharey=True)
ax0 = axes[0, 0]
ax0.pcolormesh(bx, by, np.ma.masked_invalid(obs), cmap="Greys", vmin=0.25, vmax=0.8)
ax0.set_title(
    f"Proportion observée (test)\nlog loss constante = {LL_CONST:.4f}".replace(".", ","),
    fontsize=10,
)
for ax, (nom, m) in zip(axes.ravel()[1:], modeles2.items(), strict=True):
    P = proba(m, GRILLE).reshape(XX.shape)
    im = ax.pcolormesh(
        XX, YY, P, cmap="Greys", vmin=0.25, vmax=0.8, shading="auto", rasterized=True
    )
    cs = ax.contour(
        XX,
        YY,
        P,
        levels=[0.4, 0.5, 0.6],
        colors=[ENCRE] * 3,
        linestyles=["--", "-", "--"],
        linewidths=[1, 1.6, 1],
    )
    cs.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])  # visible sur fond sombre
    ax.set_title(
        f"{nom}\nlog loss = {log_loss(y_te, proba(m, X2_te)):.4f}".replace(".", ","), fontsize=10
    )
for ax in axes.ravel():
    ax.grid(False)
axes[1, 1].set_xlabel(ETIQ2[0])
fig.supylabel(ETIQ2[1], fontsize=9)
fig.tight_layout()
fig.colorbar(im, ax=axes, label="Probabilité de victoire locale", shrink=0.8)
sauver(fig, "methodes_comparaison_surfaces")
plt.show()

La régression logistique produit des isoprobabilités parallèles ; les arbres (forêt, boosting)
des surfaces en escalier alignées sur les axes, qui suivent en partie le bruit ; le SVM à noyau
RBF une surface lisse. Avec un signal aussi faible, la flexibilité supplémentaire n'apporte pas
nécessairement de gain.

### Calibration

Un modèle est **calibré** si, parmi les matchs auxquels il attribue une probabilité $p$,
l'équipe locale gagne effectivement dans une proportion $p$. Le **diagramme de fiabilité**
compare la probabilité prédite moyenne et la fréquence observée par classe (ici, déciles de
probabilité prédite). L'**erreur de calibration espérée** (ECE) résume l'écart moyen pondéré à
la diagonale. Modèles entraînés sur les 13 variables, évalués sur les saisons de test.

In [ ]:
def fiabilite(y, p, n_classes=10):
    bornes = np.quantile(p, np.linspace(0, 1, n_classes + 1))
    idx = np.clip(np.searchsorted(bornes, p, side="right") - 1, 0, n_classes - 1)
    df = (
        pd.DataFrame({"y": np.asarray(y), "p": p, "c": idx})
        .groupby("c")
        .agg(p=("p", "mean"), f=("y", "mean"), n=("y", "size"))
    )
    ece = float(np.sum(df.n * np.abs(df.p - df.f)) / len(p))
    return df, ece


modeles = ajuster(X_tr, X_va)
p_const = np.full(len(y_te), P_CONST)
scores = [
    (
        "Probabilité constante",
        LL_CONST,
        brier_score_loss(y_te, p_const),
        0.5,
        fiabilite(y_te, p_const, 1)[1],
    )
]
fig, ax = plt.subplots(figsize=(5.4, 5.2))
ax.plot([0, 1], [0, 1], color=ENCRE_2, ls=":", marker="", lw=1, label="Calibration parfaite")
for nom, m in modeles.items():
    p = proba(m, X_te)
    df, ece = fiabilite(y_te, p)
    ax.plot(df.p, df.f, label=nom)
    scores.append((nom, log_loss(y_te, p), brier_score_loss(y_te, p), roc_auc_score(y_te, p), ece))
ax.set(
    xlabel="Probabilité prédite moyenne (décile)",
    ylabel="Fréquence observée de victoires locales",
    xlim=(0.25, 0.8),
    ylim=(0.25, 0.8),
    aspect="equal",
)
ax.legend(fontsize=8, loc="upper left")
sauver(fig, "methodes_calibration_fiabilite")
plt.show()

tab_scores = pd.DataFrame(scores, columns=["Modèle", "Log loss", "Brier", "AUC", "ECE"]).set_index(
    "Modèle"
)
sauver_tableau(tab_scores.round(4), "methodes_tab_scores_illustration")
tab_scores

Ces scores servent uniquement à illustrer les propriétés des méthodes (hyperparamètres non
optimisés, 13 variables simples). Les écarts entre modèles sont petits par rapport à l'écart
avec la probabilité constante : sur des données de hockey, l'essentiel du gain vient des
variables, et la flexibilité du modèle se paie souvent en surapprentissage. L'**AUC**
(discrimination) et la **log loss** (qualité des probabilités) ne classent pas forcément les
modèles dans le même ordre.

---
## 6. Tableau synthèse

In [ ]:
synthese = pd.DataFrame(
    [
        (
            "Forêt aléatoire",
            "Moyenne de B arbres profonds sur échantillons bootstrap "
            "et sous-ensembles de variables",
            "max_features, min_samples_leaf, B",
            "Non-linéarités et interactions, robuste, peu sensible aux "
            "hyperparamètres, parallélisable",
            "Probabilités compressées vers le centre ou bruitées "
            "(recalibration), modèle lourd, pas de dynamique",
        ),
        (
            "XGBoost",
            "Boosting d'arbres par gradient (second ordre) avec régularisation Ω(h)",
            "η, nombre d'arbres, profondeur, λ, γ, sous-échantillonnage",
            "Très performant sur données tabulaires, valeurs manquantes, régularisation riche",
            "Nombreux hyperparamètres, surapprentissage sur petits jeux, calibration à vérifier",
        ),
        (
            "LightGBM",
            "Boosting d'arbres par histogrammes, croissance par feuille, GOSS et EFB",
            "η, num_leaves, min_child_samples, nombre d'arbres",
            "Très rapide, grands jeux de données, mêmes forces que XGBoost",
            "Arbres asymétriques sujets au surapprentissage sur petits jeux, réglage délicat",
        ),
        (
            "SVM",
            "Hyperplan à marge maximale, noyaux pour la non-linéarité",
            "C, noyau et γ",
            "Base théorique solide, convexe, efficace en grande dimension",
            "Pas de probabilité native (Platt), marge peu adaptée aux "
            "classes chevauchantes, coût O(n²–n³)",
        ),
    ],
    columns=["Méthode", "Principe", "Hyperparamètres principaux", "Avantages", "Limites"],
).set_index("Méthode")
sauver_tableau(synthese, "methodes_tab_apprentissage_automatique")
synthese